# Notebook 01: Data Analysis, Stratified Split & Preprocessing

**Retail Shelf Detection -- ShelfVision AI**

This notebook performs:
1. Dataset structure validation
2. Class distribution analysis across all splits
3. Class imbalance detection and statistics
4. Stratified re-split (80/10/10) to fix class distribution
5. Optional oversampling of minority classes
6. Before/after comparison visualization

**Dataset**: Upload as Kaggle Dataset or use Google Drive


## Cell 1: Setup & Dataset Validation


In [ ]:
!pip install ultralytics>=8.3.0 scikit-learn albumentations -q

import yaml
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

# ---- CONFIGURE YOUR DATASET PATH HERE ----
# Kaggle:
# DATASET_DIR = Path("/kaggle/input/retail-shelf-dataset")
# WORKING_DIR = Path("/kaggle/working")

# Google Colab (after mounting drive):
DATASET_DIR = Path("/content/drive/MyDrive/yolov11-object-detection/dataset")
WORKING_DIR = Path("/content/working")

WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Load class names from data.yaml
with open(DATASET_DIR / "data.yaml", 'r') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['names']
NUM_CLASSES = cfg['nc']

# Validate dataset structure
print("=" * 65)
print("  DATASET VALIDATION")
print("=" * 65)

for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    imgs = list(img_dir.glob("*.jpg")) if img_dir.exists() else []
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    status = "OK" if len(imgs) > 0 and abs(len(imgs) - len(lbls)) < 5 else "ERROR"
    print(f"  [{status}] {split}: {len(imgs)} images, {len(lbls)} labels")

print(f"\n  Classes: {NUM_CLASSES}")
print(f"  Names: {CLASS_NAMES[:5]}... (showing first 5)")
print("=" * 65)

## Cell 2: Class Distribution Analysis


In [ ]:
def count_labels(label_dir):
    """Count instances of each class in a label directory."""
    counts = Counter()
    total_objects = 0
    total_images = 0
    empty_images = 0

    label_dir = Path(label_dir)
    if not label_dir.exists():
        return counts, total_objects, total_images, empty_images

    for label_file in sorted(label_dir.glob("*.txt")):
        total_images += 1
        file_has_objects = False
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    counts[cls_id] += 1
                    total_objects += 1
                    file_has_objects = True
        if not file_has_objects:
            empty_images += 1

    return counts, total_objects, total_images, empty_images


# Count for each split
splits = {
    'train': DATASET_DIR / "train" / "labels",
    'valid': DATASET_DIR / "valid" / "labels",
    'test':  DATASET_DIR / "test"  / "labels",
}

all_counts = {}
for split_name, label_dir in splits.items():
    counts, total_obj, total_img, empty = count_labels(label_dir)
    all_counts[split_name] = counts
    print(f"\n{split_name.upper()}:")
    print(f"   Images: {total_img} | Objects: {total_obj} | Empty: {empty}")
    print(f"   Avg objects/image: {total_obj / max(total_img, 1):.1f}")

# Build full DataFrame
rows = []
for cls_id in range(NUM_CLASSES):
    cls_name = CLASS_NAMES[cls_id]
    train_count = all_counts['train'].get(cls_id, 0)
    valid_count = all_counts['valid'].get(cls_id, 0)
    test_count  = all_counts['test'].get(cls_id, 0)
    total = train_count + valid_count + test_count
    rows.append({
        'class_id': cls_id, 'class_name': cls_name,
        'train': train_count, 'valid': valid_count,
        'test': test_count, 'total': total,
    })

df = pd.DataFrame(rows).sort_values('total', ascending=False).reset_index(drop=True)
total_all = df['total'].sum()

# Key statistics
max_cls = df.iloc[0]
min_cls = df[df['total'] > 0].iloc[-1]
ratio = max_cls['total'] / max(min_cls['total'], 1)
zero_train = len(df[df['train'] == 0])

print(f"\n{'='*55}")
print(f"  IMBALANCE STATISTICS")
print(f"{'='*55}")
print(f"  Most common:   {max_cls['class_name']} ({max_cls['total']} instances)")
print(f"  Least common:  {min_cls['class_name']} ({min_cls['total']} instances)")
print(f"  Imbalance ratio: {ratio:.1f}x")
print(f"  Zero-train classes: {zero_train}")
for t in [5, 10, 20]:
    print(f"  Classes with < {t} TRAIN samples: {len(df[df['train'] < t])}/{NUM_CLASSES}")
print(f"{'='*55}")

df.to_csv(WORKING_DIR / "class_distribution.csv", index=False)
print(f"\nSaved: class_distribution.csv")

## Cell 3: Visualizations


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# 1. Horizontal bar -- all classes
ax1 = axes[0, 0]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(df)))[::-1]
ax1.barh(range(len(df)), df['total'].values, color=colors, edgecolor='white', linewidth=0.3)
ax1.set_yticks(range(len(df)))
ax1.set_yticklabels(df['class_name'].values, fontsize=7)
ax1.invert_yaxis()
ax1.set_xlabel("Total Instances", fontweight='bold')
ax1.set_title("Class Distribution (All Splits)", fontweight='bold')
ax1.axvline(x=df['total'].mean(), color='red', linestyle='--', alpha=0.7, label=f"Mean: {df['total'].mean():.0f}")
ax1.legend(fontsize=9)
ax1.grid(axis='x', alpha=0.3)

# 2. Split comparison
ax2 = axes[0, 1]
x = np.arange(len(df))
w = 0.3
ax2.bar(x - w, df['train'], w, label='Train', color='#3498db', alpha=0.8)
ax2.bar(x, df['valid'], w, label='Valid', color='#2ecc71', alpha=0.8)
ax2.bar(x + w, df['test'], w, label='Test', color='#e74c3c', alpha=0.8)
ax2.set_xlabel("Class (sorted by total)", fontweight='bold')
ax2.set_ylabel("Instances", fontweight='bold')
ax2.set_title("Split Distribution per Class", fontweight='bold')
ax2.set_xticks(x[::5])
ax2.set_xticklabels(df['class_name'].values[::5], rotation=45, ha='right', fontsize=7)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# 3. Box plot
ax3 = axes[1, 0]
bp = ax3.boxplot([df['train'], df['valid'], df['test'], df['total']],
                  labels=['Train', 'Valid', 'Test', 'Total'], patch_artist=True)
for patch, c in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax3.set_ylabel("Instances per Class", fontweight='bold')
ax3.set_title("Distribution Spread", fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# 4. Pareto
ax4 = axes[1, 1]
cumulative = np.cumsum(df['total'].values) / df['total'].sum() * 100
ax4.fill_between(range(len(cumulative)), cumulative, alpha=0.3, color='#3498db')
ax4.plot(range(len(cumulative)), cumulative, 'o-', color='#2980b9', markersize=4)
ax4.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% of data')
ax4.set_xlabel("Number of Classes (sorted)", fontweight='bold')
ax4.set_ylabel("Cumulative %", fontweight='bold')
ax4.set_title("Cumulative Distribution (Pareto)", fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(WORKING_DIR / "class_imbalance_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved!")

## Cell 4: Stratified Split (80/10/10)

The original dataset split has **4 classes with zero training samples** and an unbalanced
distribution across splits. Instead of the rescue/oversample approach, we pool ALL images
and re-split using stratified sampling to ensure proportional representation.


In [ ]:
# Pool ALL images from ALL splits
all_images = []
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    if not img_dir.exists():
        continue
    for img_path in sorted(img_dir.glob("*.jpg")):
        stem = img_path.stem
        lbl_path = lbl_dir / f"{stem}.txt"
        unique_key = f"{split}_{stem}"

        annotations = []
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        annotations.append(int(parts[0]))

        if annotations:
            all_images.append({
                'key': unique_key,
                'img_path': img_path,
                'lbl_path': lbl_path,
                'classes': annotations,
                'dominant_class': Counter(annotations).most_common(1)[0][0],
            })

print(f"Total pooled images: {len(all_images)}")

# Identify rare classes (< 3 images as dominant) -- force into train
dominant_counts = Counter(img['dominant_class'] for img in all_images)
rare_classes = {cls for cls, count in dominant_counts.items() if count < 3}
rare_images = [img for img in all_images if img['dominant_class'] in rare_classes]
main_images = [img for img in all_images if img['dominant_class'] not in rare_classes]

print(f"Rare classes (forced to train): {len(rare_classes)}")
print(f"Main images to split: {len(main_images)}")

# First split: train (80%) vs temp (20%)
main_keys = [img['key'] for img in main_images]
main_dominant = [img['dominant_class'] for img in main_images]

train_keys, temp_keys, train_dom, temp_dom = train_test_split(
    main_keys, main_dominant,
    test_size=0.20,
    stratify=main_dominant,
    random_state=42
)

# Second split: handle singletons in temp
temp_class_counts = Counter(temp_dom)
singleton_classes = {cls for cls, cnt in temp_class_counts.items() if cnt < 2}

if singleton_classes:
    print(f"{len(singleton_classes)} singleton classes in temp -- assigning alternately")
    temp_main_keys, temp_main_dom, singleton_keys = [], [], []
    for key, dom in zip(temp_keys, temp_dom):
        if dom in singleton_classes:
            singleton_keys.append(key)
        else:
            temp_main_keys.append(key)
            temp_main_dom.append(dom)

    if len(temp_main_keys) > 1:
        valid_keys, test_keys = train_test_split(
            temp_main_keys, test_size=0.5, stratify=temp_main_dom, random_state=42
        )
    else:
        valid_keys, test_keys = temp_main_keys, []

    for i, key in enumerate(singleton_keys):
        (valid_keys if i % 2 == 0 else test_keys).append(key)
else:
    valid_keys, test_keys = train_test_split(
        temp_keys, test_size=0.5, stratify=temp_dom, random_state=42
    )

# Add rare images to train
train_keys.extend(img['key'] for img in rare_images)

# Build key -> split mapping
split_map = {}
for k in train_keys: split_map[k] = 'train'
for k in valid_keys: split_map[k] = 'valid'
for k in test_keys:  split_map[k] = 'test'

print(f"\nStratified Split:")
print(f"  Train: {len(train_keys)} ({len(train_keys)/len(all_images)*100:.1f}%)")
print(f"  Valid: {len(valid_keys)} ({len(valid_keys)/len(all_images)*100:.1f}%)")
print(f"  Test:  {len(test_keys)} ({len(test_keys)/len(all_images)*100:.1f}%)")

## Cell 5: Write Stratified Dataset to Disk


In [ ]:
STRATIFIED_DIR = WORKING_DIR / "stratified_dataset"

if STRATIFIED_DIR.exists():
    shutil.rmtree(STRATIFIED_DIR)

for split in ['train', 'valid', 'test']:
    (STRATIFIED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (STRATIFIED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

copied = {'train': 0, 'valid': 0, 'test': 0}
for img_info in all_images:
    split = split_map[img_info['key']]
    dst_name = img_info['key'].replace('/', '_')
    shutil.copy2(img_info['img_path'], STRATIFIED_DIR / split / "images" / f"{dst_name}.jpg")
    shutil.copy2(img_info['lbl_path'], STRATIFIED_DIR / split / "labels" / f"{dst_name}.txt")
    copied[split] += 1

print("Stratified dataset created:")
for split, count in copied.items():
    print(f"  {split}: {count} images")

## Cell 6: Oversample Minority Classes (Optional)


In [ ]:
TARGET_MIN = 15
TRAIN_IMG_DIR = STRATIFIED_DIR / "train" / "images"
TRAIN_LBL_DIR = STRATIFIED_DIR / "train" / "labels"

train_counts = Counter()
class_to_files = defaultdict(list)
for lbl_path in TRAIN_LBL_DIR.glob("*.txt"):
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                train_counts[cls_id] += 1
                class_to_files[cls_id].append(lbl_path.stem)

total_os = 0
for cls_id in range(NUM_CLASSES):
    cur = train_counts.get(cls_id, 0)
    if cur >= TARGET_MIN or cur == 0:
        continue
    files = list(set(class_to_files[cls_id]))
    if not files:
        continue
    need = TARGET_MIN - cur
    avg_per = cur / len(files) if files else 1
    n_copies = max(1, int(np.ceil(need / avg_per)))
    for i in range(n_copies):
        src = random.choice(files)
        if (TRAIN_IMG_DIR / f"{src}.jpg").exists():
            shutil.copy2(TRAIN_IMG_DIR / f"{src}.jpg", TRAIN_IMG_DIR / f"os_{cls_id}_{i}_{src}.jpg")
            shutil.copy2(TRAIN_LBL_DIR / f"{src}.txt", TRAIN_LBL_DIR / f"os_{cls_id}_{i}_{src}.txt")
            total_os += 1

print(f"Oversampled: {total_os} images")
print(f"Final train images: {len(list(TRAIN_IMG_DIR.glob('*.jpg')))}")

## Cell 7: Before/After Comparison


In [ ]:
# Count original train vs stratified train
original_train = {r['class_id']: r['train'] for _, r in df.iterrows()}

strat_train_counts = Counter()
for lbl_path in TRAIN_LBL_DIR.glob("*.txt"):
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                strat_train_counts[int(parts[0])] += 1

rows_cmp = []
for cls_id in range(NUM_CLASSES):
    rows_cmp.append({
        'class': CLASS_NAMES[cls_id],
        'before': original_train.get(cls_id, 0),
        'after': strat_train_counts.get(cls_id, 0),
    })
df_cmp = pd.DataFrame(rows_cmp)

fig, axes = plt.subplots(1, 2, figsize=(22, 14))

for ax, col, title, color_label in [
    (axes[0], 'before', 'BEFORE: Original Split', '#e74c3c'),
    (axes[1], 'after', 'AFTER: Stratified + Oversample', '#2ecc71'),
]:
    sorted_df = df_cmp.sort_values(col, ascending=True)
    colors = ['#e74c3c' if v == 0 else '#f39c12' if v < 15 else '#2ecc71' for v in sorted_df[col]]
    ax.barh(range(len(sorted_df)), sorted_df[col], color=colors, edgecolor='white', linewidth=0.3)
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['class'], fontsize=7)
    ax.set_xlabel("Training Instances", fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', color=color_label)
    ax.axvline(x=15, color='red', linestyle='--', alpha=0.5, label='Target min (15)')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle("Original vs Stratified Split: Training Distribution per Class",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(WORKING_DIR / "stratified_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary
zero_before = sum(1 for r in rows_cmp if r['before'] == 0)
zero_after = sum(1 for r in rows_cmp if r['after'] == 0)
print(f"\nZero-train classes: {zero_before} (before) -> {zero_after} (after)")
print(f"Comparison chart saved!")

## Cell 8: Export Stratified Dataset


In [ ]:
# Create data.yaml for the stratified dataset
data_yaml = {
    'train': str(STRATIFIED_DIR / "train" / "images"),
    'val': str(STRATIFIED_DIR / "valid" / "images"),
    'test': str(STRATIFIED_DIR / "test" / "images"),
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES,
}

DATA_YAML = WORKING_DIR / "data_stratified.yaml"
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml saved: {DATA_YAML}")
print(f"\nStratified dataset is ready for training in Notebook 02.")
print(f"Location: {STRATIFIED_DIR}")